In [2]:
import pandas as pd
import numpy as np
import os
import sys
import matplotlib.pyplot as plt
from pathlib import Path

import sys, subprocess, importlib
print('Notebook Python executable:', sys.executable) # to check the python environment and the path from the local system
importlib.import_module('matplotlib')
import seaborn as sns

import warnings
warnings.filterwarnings('ignore')

Notebook Python executable: <PROJECT_ROOT>\.venv\Scripts\python.exe


In [3]:
# Resolve CSV path robustly for notebook runs from different working directories
current_dir = Path.cwd()
project_root = current_dir if (current_dir / "data").exists() else current_dir.parent

candidate_paths = [
    Path("survey.csv"),
    Path("data") / "survey.csv",
    Path("..") / "data" / "survey.csv",
    current_dir / "data" / "survey.csv",
    project_root / "data" / "survey.csv",
]

csv_path = next((p for p in candidate_paths if p.exists()), None)
if csv_path is None:
    checked = "\n".join(str(p) for p in candidate_paths)
    raise FileNotFoundError(f"Could not find survey.csv. Checked:\n{checked}")

# Detect delimiter from the header row (supports ';' and ',').
with open(csv_path, "r", encoding="ISO-8859-1") as f:
    header_line = f.readline()
sep = ";" if header_line.count(";") > header_line.count(",") else ","

# Read CSV using first row as header
df_survey = pd.read_csv(
    csv_path,
    encoding="ISO-8859-1",
    low_memory=False,
    header=0,
    sep=sep,
    skipinitialspace=True,
    on_bad_lines="skip",
)

# Normalize column names: lowercase + snake_case
df_survey.columns = (
    df_survey.columns
    .str.strip()
    .str.lower()
    .str.replace(r"[^a-z0-9]+", "_", regex=True)
    .str.strip("_")
)

print("\n" + "=" * 60)
print("SURVEY TABLE - Initial Assessment")
print("=" * 60)
print(df_survey.head(5))

# print("\nNormalized column names:")
# print(df_survey.columns.tolist())


SURVEY TABLE - Initial Assessment
          timestamp  age  gender         country state self_employed  \
0  27/08/2014 11:29   37  Female   United States    IL           NaN   
1  27/08/2014 11:29   44       M   United States    IN           NaN   
2  27/08/2014 11:29   32    Male          Canada   NaN           NaN   
3  27/08/2014 11:29   31    Male  United Kingdom   NaN           NaN   
4  27/08/2014 11:30   31    Male   United States    TX           NaN   

  family_history treatment work_interfere    no_employees  ...  \
0             No       Yes          Often          Jun-25  ...   
1             No        No         Rarely  More than 1000  ...   
2             No        No         Rarely          Jun-25  ...   
3            Yes       Yes          Often          26-100  ...   
4             No        No          Never         100-500  ...   

                leave mental_health_consequence phys_health_consequence  \
0       Somewhat easy                        No             

In [4]:
pick_missing_values_count = df_survey.isnull().sum()
print("Missing values in df_survey:\n")
print(pick_missing_values_count,"\n")

Missing values in df_survey:

timestamp                       0
age                             0
gender                          0
country                         0
state                         515
self_employed                  18
family_history                  0
treatment                       0
work_interfere                264
no_employees                    0
remote_work                     0
tech_company                    0
benefits                        0
care_options                    0
wellness_program                0
seek_help                       0
anonymity                       0
leave                           0
mental_health_consequence       0
phys_health_consequence         0
coworkers                       0
supervisor                      0
mental_health_interview         0
phys_health_interview           0
mental_vs_physical              0
obs_consequence                 0
comments                     1096
dtype: int64 



In [5]:
df_survey["comments"] = df_survey["comments"].fillna("No comments")

In [6]:
# df_survey["work_interfere"] = df_survey["work_interfere"].fillna("Never")

In [7]:
df_survey["self_employed"] = df_survey["self_employed"].fillna("No")

In [8]:
print("Unique countries:")
print(df_survey["country"].unique())

print("\nTotal unique countries:", df_survey["country"].nunique())

Unique countries:
['United States' 'Canada' 'United Kingdom' 'Bulgaria' 'France' 'Portugal'
 'Netherlands' 'Switzerland' 'Poland' 'Australia' 'Germany' 'Russia'
 'Mexico' 'Brazil' 'Slovenia' 'Costa Rica' 'Austria' 'Ireland' 'India'
 'South Africa' 'Italy' 'Sweden' 'Colombia' 'Latvia' 'Romania' 'Belgium'
 'New Zealand' 'Zimbabwe' 'Spain' 'Finland' 'Uruguay' 'Israel'
 'Bosnia and Herzegovina' 'Hungary' 'Singapore' 'Japan' 'Nigeria'
 'Croatia' 'Norway' 'Thailand' 'Denmark' 'Bahamas, The' 'Greece' 'Moldova'
 'Georgia' 'China' 'Czech Republic' 'Philippines']

Total unique countries: 48


In [9]:
country_counts = df_survey["country"].value_counts(dropna=False)

display(country_counts.reset_index().rename(
    columns={"index": "Country", "count": "Respondent Count"}
))

,country,Respondent Count
0,United States,751
1,United Kingdom,185
2,Canada,72
3,Germany,45
4,Netherlands,27
5,Ireland,27
6,Australia,21
7,France,13
8,India,10
9,New Zealand,8


In [10]:
df_survey["state"] = df_survey["state"].fillna("Not Available")

In [11]:
pick_missing_values_count = df_survey.isnull().sum()
print("Missing values in df_survey:\n")
print(pick_missing_values_count,"\n")

Missing values in df_survey:

timestamp                      0
age                            0
gender                         0
country                        0
state                          0
self_employed                  0
family_history                 0
treatment                      0
work_interfere               264
no_employees                   0
remote_work                    0
tech_company                   0
benefits                       0
care_options                   0
wellness_program               0
seek_help                      0
anonymity                      0
leave                          0
mental_health_consequence      0
phys_health_consequence        0
coworkers                      0
supervisor                     0
mental_health_interview        0
phys_health_interview          0
mental_vs_physical             0
obs_consequence                0
comments                       0
dtype: int64 



In [12]:
gender_counts = df_survey["gender"].value_counts(dropna=False)

display(gender_counts.reset_index().rename(
    columns={"index": "Gender", "count": "Respondent Count"}
))

,gender,Respondent Count
0,Male,615
1,male,206
2,Female,121
3,M,116
4,female,62
5,F,38
6,m,34
7,f,15
8,Make,4
9,Male,3


### Gender Standardisation

The original Gender column contains inconsistent responses due to differences in
capitalisation, abbreviations, spelling and self-described gender identities.

To improve consistency, the responses are standardised into three categories:
- Male
- Female
- Others

Ambiguous and non-binary gender descriptions are grouped under Others.
The original responses are preserved in the raw dataset.

In [13]:
# Define gender mapping
gender_mapping = {
    # Male
    "male": "Male",
    "m": "Male",
    "man": "Male",
    "make": "Male",
    "cis male": "Male",
    "male-ish": "Male",
    "male (cis)": "Male",
    "mal": "Male",
    "maile": "Male",
    "mail": "Male",
    "cis man": "Male",

    # Female
    "female": "Female",
    "f": "Female",
    "woman": "Female",
    "femake": "Female",
    "cis female": "Female",
    "female (cis)": "Female",
    "female (trans)": "Female",
    "trans-female": "Female",
    "trans woman": "Female",
    "cis-female/femme": "Female",
    "femal": "Female",

    # Others
    "genderqueer": "Others",
    "non-binary": "Others",
    "enby": "Others",
    "fluid": "Others",
    "agender": "Others",
    "androgyne": "Others",
    "queer": "Others",
    "neuter": "Others",
    "nah": "Others",
    "all": "Others",
    "p": "Others",
    "something kinda male?": "Others",
    "ostensibly male, unsure what that really means": "Others",
    "a little about you": "Others"
}

# Standardise gender responses
df_survey["gender"] = (
    df_survey["gender"]
    .astype("string")
    .str.strip()
    .str.lower()
    .map(gender_mapping)
    .fillna("Others")
)

# Display standardised gender distribution
gender_standardised = (
    df_survey["gender"]
    .value_counts(dropna=False)
    .rename_axis("Gender")
    .reset_index(name="Respondent Count")
)

display(gender_standardised)

# Validate total records
print("\nOriginal dataset rows:", len(df_survey))
print("Total standardised responses:", gender_standardised["Respondent Count"].sum())
print("Missing gender values:", df_survey["gender"].isna().sum())

,Gender,Respondent Count
0,Male,989
1,Female,250
2,Others,20



Original dataset rows: 1259
Total standardised responses: 1259
Missing gender values: 0


In [14]:
# Inspect all unique employee-size responses

employee_counts = (
    df_survey["no_employees"]
    .value_counts(dropna=False)
    .rename_axis("Employee Size")
    .reset_index(name="Respondent Count")
)

display(employee_counts)

print(
    "Total records:",
    employee_counts["Respondent Count"].sum()
)

,Employee Size,Respondent Count
0,Jun-25,290
1,26-100,289
2,More than 1000,282
3,100-500,176
4,01-May,162
5,500-1000,60


Total records: 1259


In [15]:
# Restore employee-size categories incorrectly converted to dates

employee_size_mapping = {
    "Jun-25": "6-25",
    "01-May": "1-5"
}

df_survey["no_employees"] = (
    df_survey["no_employees"]
    .astype("string")
    .str.strip()
    .replace(employee_size_mapping)
)

# Verify the corrected distribution
employee_counts = (
    df_survey["no_employees"]
    .value_counts(dropna=False)
    .rename_axis("Employee Size")
    .reset_index(name="Respondent Count")
)

display(employee_counts)

print("\nTotal records:", len(df_survey))

,Employee Size,Respondent Count
0,6-25,290
1,26-100,289
2,More than 1000,282
3,100-500,176
4,1-5,162
5,500-1000,60



Total records: 1259


In [16]:
df_survey.head(5)

,timestamp,age,gender,country,state,self_employed,family_history,treatment,work_interfere,no_employees,...,leave,mental_health_consequence,phys_health_consequence,coworkers,supervisor,mental_health_interview,phys_health_interview,mental_vs_physical,obs_consequence,comments
0,27/08/2014 11:29,37,Female,United States,IL,No,No,Yes,Often,6-25,...,Somewhat easy,No,No,Some of them,Yes,No,Maybe,Yes,No,No comments
1,27/08/2014 11:29,44,Male,United States,IN,No,No,No,Rarely,More than 1000,...,Don't know,Maybe,No,No,No,No,No,Don't know,No,No comments
2,27/08/2014 11:29,32,Male,Canada,Not Available,No,No,No,Rarely,6-25,...,Somewhat difficult,No,No,Yes,Yes,Yes,Yes,No,No,No comments
3,27/08/2014 11:29,31,Male,United Kingdom,Not Available,No,Yes,Yes,Often,26-100,...,Somewhat difficult,Yes,Yes,Some of them,No,Maybe,Maybe,No,Yes,No comments
4,27/08/2014 11:30,31,Male,United States,TX,No,No,No,Never,100-500,...,Don't know,No,No,Some of them,Yes,Yes,Yes,Don't know,No,No comments


In [17]:
df_survey.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1259 entries, 0 to 1258
Data columns (total 27 columns):
 #   Column                     Non-Null Count  Dtype 
---  ------                     --------------  ----- 
 0   timestamp                  1259 non-null   object
 1   age                        1259 non-null   int64 
 2   gender                     1259 non-null   object
 3   country                    1259 non-null   object
 4   state                      1259 non-null   object
 5   self_employed              1259 non-null   object
 6   family_history             1259 non-null   object
 7   treatment                  1259 non-null   object
 8   work_interfere             995 non-null    object
 9   no_employees               1259 non-null   string
 10  remote_work                1259 non-null   object
 11  tech_company               1259 non-null   object
 12  benefits                   1259 non-null   object
 13  care_options               1259 non-null   object
 14  wellness

In [18]:
# Display all columns without truncation
pd.set_option("display.max_columns", None)

# Display more rows
pd.set_option("display.max_rows", 100)

# Allow wider DataFrame display
pd.set_option("display.width", None)

# Prevent column content from being shortened
pd.set_option("display.max_colwidth", None)

In [19]:
display(df_survey.head(10))

,timestamp,age,gender,country,state,self_employed,family_history,treatment,work_interfere,no_employees,remote_work,tech_company,benefits,care_options,wellness_program,seek_help,anonymity,leave,mental_health_consequence,phys_health_consequence,coworkers,supervisor,mental_health_interview,phys_health_interview,mental_vs_physical,obs_consequence,comments
0,27/08/2014 11:29,37,Female,United States,IL,No,No,Yes,Often,6-25,No,Yes,Yes,Not sure,No,Yes,Yes,Somewhat easy,No,No,Some of them,Yes,No,Maybe,Yes,No,No comments
1,27/08/2014 11:29,44,Male,United States,IN,No,No,No,Rarely,More than 1000,No,No,Don't know,No,Don't know,Don't know,Don't know,Don't know,Maybe,No,No,No,No,No,Don't know,No,No comments
2,27/08/2014 11:29,32,Male,Canada,Not Available,No,No,No,Rarely,6-25,No,Yes,No,No,No,No,Don't know,Somewhat difficult,No,No,Yes,Yes,Yes,Yes,No,No,No comments
3,27/08/2014 11:29,31,Male,United Kingdom,Not Available,No,Yes,Yes,Often,26-100,No,Yes,No,Yes,No,No,No,Somewhat difficult,Yes,Yes,Some of them,No,Maybe,Maybe,No,Yes,No comments
4,27/08/2014 11:30,31,Male,United States,TX,No,No,No,Never,100-500,Yes,Yes,Yes,No,Don't know,Don't know,Don't know,Don't know,No,No,Some of them,Yes,Yes,Yes,Don't know,No,No comments
5,27/08/2014 11:31,33,Male,United States,TN,No,Yes,No,Sometimes,6-25,No,Yes,Yes,Not sure,No,Don't know,Don't know,Don't know,No,No,Yes,Yes,No,Maybe,Don't know,No,No comments
6,27/08/2014 11:31,35,Female,United States,MI,No,Yes,Yes,Sometimes,1-5,Yes,Yes,No,No,No,No,No,Somewhat difficult,Maybe,Maybe,Some of them,No,No,No,Don't know,No,No comments
7,27/08/2014 11:32,39,Male,Canada,Not Available,No,No,No,Never,1-5,Yes,Yes,No,Yes,No,No,Yes,Don't know,No,No,No,No,No,No,No,No,No comments
8,27/08/2014 11:32,42,Female,United States,IL,No,Yes,Yes,Sometimes,100-500,No,Yes,Yes,Yes,No,No,No,Very difficult,Maybe,No,Yes,Yes,No,Maybe,No,No,No comments
9,27/08/2014 11:32,23,Male,Canada,Not Available,No,No,No,Never,26-100,No,Yes,Don't know,No,Don't know,Don't know,Don't know,Don't know,No,No,Yes,Yes,Maybe,Maybe,Yes,No,No comments


In [20]:
column_summary = pd.DataFrame({
    "Column": df_survey.columns,
    "Data Type": df_survey.dtypes.astype(str).values,
    "Unique Values": df_survey.nunique(dropna=False).values,
    "Missing Values": df_survey.isna().sum().values
})

display(column_summary)

,Column,Data Type,Unique Values,Missing Values
0,timestamp,object,884,0
1,age,int64,53,0
2,gender,object,3,0
3,country,object,48,0
4,state,object,46,0
5,self_employed,object,2,0
6,family_history,object,2,0
7,treatment,object,2,0
8,work_interfere,object,5,264
9,no_employees,string,6,0


## Data Type Conversion

The dataset contains numerical, categorical, datetime and free-text variables.

The data types are adjusted according to the nature of each variable to improve data consistency and prepare the dataset for subsequent machine learning operations.

Ordinal variables are assigned an explicit category order to preserve their natural ranking.

Target encoding, categorical encoding and feature scaling will be performed after the train-test split.

In [21]:
# 1. Convert timestamp to datetime
df_survey["timestamp"] = pd.to_datetime(
    df_survey["timestamp"],
    format="%d/%m/%Y %H:%M",
    errors="coerce"
)

# 2. Convert age to nullable integer
df_survey["age"] = pd.to_numeric(
    df_survey["age"],
    errors="coerce"
).astype("Int64")

# 3. Define ordinal category orders
ordinal_categories = {
    "work_interfere": [
        "Never",
        "Rarely",
        "Sometimes",
        "Often"
    ],

    "no_employees": [
        "1-5",
        "6-25",
        "26-100",
        "100-500",
        "500-1000",
        "More than 1000"
    ],

    "leave": [
        "Very easy",
        "Somewhat easy",
        "Don't know",
        "Somewhat difficult",
        "Very difficult"
    ]
}

# 4. Convert ordinal columns
for column, categories in ordinal_categories.items():
    df_survey[column] = pd.Categorical(
        df_survey[column],
        categories=categories,
        ordered=True
    )

# 5. Convert remaining categorical columns
categorical_columns = [
    "gender",
    "country",
    "state",
    "self_employed",
    "family_history",
    "treatment",
    "remote_work",
    "tech_company",
    "benefits",
    "care_options",
    "wellness_program",
    "seek_help",
    "anonymity",
    "mental_health_consequence",
    "phys_health_consequence",
    "coworkers",
    "supervisor",
    "mental_health_interview",
    "phys_health_interview",
    "mental_vs_physical",
    "obs_consequence"
]

for column in categorical_columns:
    df_survey[column] = df_survey[column].astype("category")

# 6. Convert comments to string
df_survey["comments"] = df_survey["comments"].astype("string")

print("Data type conversion completed.")

Data type conversion completed.


In [22]:
dtype_summary = pd.DataFrame({
    "Column": df_survey.columns,
    "Data Type": df_survey.dtypes.astype(str).values,
    "Unique Values": df_survey.nunique(dropna=False).values,
    "Missing Values": df_survey.isna().sum().values
})

display(dtype_summary)

df_survey.info()

,Column,Data Type,Unique Values,Missing Values
0,timestamp,datetime64[ns],884,0
1,age,Int64,53,0
2,gender,category,3,0
3,country,category,48,0
4,state,category,46,0
5,self_employed,category,2,0
6,family_history,category,2,0
7,treatment,category,2,0
8,work_interfere,category,5,264
9,no_employees,category,6,0


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1259 entries, 0 to 1258
Data columns (total 27 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   timestamp                  1259 non-null   datetime64[ns]
 1   age                        1259 non-null   Int64         
 2   gender                     1259 non-null   category      
 3   country                    1259 non-null   category      
 4   state                      1259 non-null   category      
 5   self_employed              1259 non-null   category      
 6   family_history             1259 non-null   category      
 7   treatment                  1259 non-null   category      
 8   work_interfere             995 non-null    category      
 9   no_employees               1259 non-null   category      
 10  remote_work                1259 non-null   category      
 11  tech_company               1259 non-null   category      
 12  benefi

In [23]:
# Standardise country names

country_mapping = {
    "Bahamas, The": "Bahamas"
}

df_survey["country"] = (
    df_survey["country"]
    .replace(country_mapping)
)

# Verify
print(df_survey["country"].value_counts().head(20))

country
United States     751
United Kingdom    185
Canada             72
Germany            45
Ireland            27
Netherlands        27
Australia          21
France             13
India              10
New Zealand         8
Italy               7
Sweden              7
Poland              7
Switzerland         7
Belgium             6
Brazil              6
South Africa        6
Israel              5
Bulgaria            4
Singapore           4
Name: count, dtype: int64


In [24]:
# Identify invalid age records using the 16-100 range

invalid_age_mask = (
    (df_survey["age"] < 16) |
    (df_survey["age"] > 100) |
    (df_survey["age"].isna())
)

# Preserve affected records for inspection
invalid_age_records = df_survey.loc[invalid_age_mask].copy()

print("Number of invalid or missing age records:",
      len(invalid_age_records))

display(
    invalid_age_records[
        ["age", "gender", "country", "treatment",
         "no_employees", "comments"]
    ]
)

Number of invalid or missing age records: 8


,age,gender,country,treatment,no_employees,comments
143,-29,Male,United States,No,More than 1000,No comments
364,329,Male,United States,Yes,6-25,No comments
390,99999999999,Others,Zimbabwe,Yes,1-5,No comments
715,-1726,Male,United Kingdom,Yes,26-100,No comments
734,5,Male,United States,No,100-500,We had a developer suffer from depression and pretty hard burnout but he refused treatment even when the company said we'd foot the bill. Eventually he had to be asked to resign which was a shame. I don't know if we have any specific programs for mental health but we're definitely on the lookout for those types of issues.
989,8,Others,Bahamas,Yes,1-5,No comments
1090,11,Male,United States,No,1-5,No comments
1127,-1,Others,United States,Yes,1-5,password: testered


In [25]:
# Handle invalid age values

# Define valid age range
MIN_AGE = 16
MAX_AGE = 100

# Identify invalid ages
invalid_age_mask = (
    (df_survey["age"] < MIN_AGE) |
    (df_survey["age"] > MAX_AGE)
)

# Count invalid records before cleaning
invalid_age_count = invalid_age_mask.sum()

# Replace invalid ages with missing values
df_survey.loc[invalid_age_mask, "age"] = pd.NA

# Validate results
print("AGE CLEANING SUMMARY")
print("-" * 40)
print("Invalid ages corrected:", invalid_age_count)
print("Missing age values:", df_survey["age"].isna().sum())
print("Remaining invalid ages:", (
    (df_survey["age"] < MIN_AGE) |
    (df_survey["age"] > MAX_AGE)
).sum())
print("Total dataset rows:", len(df_survey))
print("Age data type:", df_survey["age"].dtype)

# Display age distribution
display(
    df_survey["age"]
    .value_counts(dropna=False)
    .sort_index()
    .rename_axis("Age")
    .reset_index(name="Respondent Count")
)

AGE CLEANING SUMMARY
----------------------------------------
Invalid ages corrected: 8
Missing age values: 8
Remaining invalid ages: 0
Total dataset rows: 1259
Age data type: Int64


,Age,Respondent Count
0,18,7
1,19,9
2,20,6
3,21,16
4,22,21
5,23,51
6,24,46
7,25,61
8,26,75
9,27,71


In [26]:
# Filter records where age is missing
missing_age_records = df_survey[df_survey["age"].isna()].copy()

# Display the number of records
print("Records with missing age:", len(missing_age_records))

# Display all columns, with comments shortened for readability
display(
    missing_age_records.assign(
        comments=missing_age_records["comments"].astype("string").str.slice(0, 50)
    )
)

Records with missing age: 8


,timestamp,age,gender,country,state,self_employed,family_history,treatment,work_interfere,no_employees,remote_work,tech_company,benefits,care_options,wellness_program,seek_help,anonymity,leave,mental_health_consequence,phys_health_consequence,coworkers,supervisor,mental_health_interview,phys_health_interview,mental_vs_physical,obs_consequence,comments
143,2014-08-27 12:39:00,<NA>,Male,United States,MN,No,No,No,NaN,More than 1000,Yes,No,Yes,No,Don't know,Yes,Don't know,Don't know,No,No,Some of them,Yes,No,No,Don't know,No,No comments
364,2014-08-27 15:05:00,<NA>,Male,United States,OH,No,No,Yes,Often,6-25,Yes,Yes,Yes,Yes,No,No,Don't know,Don't know,Maybe,No,Some of them,No,No,No,No,No,No comments
390,2014-08-27 15:24:00,<NA>,Others,Zimbabwe,Not Available,Yes,Yes,Yes,Often,1-5,No,Yes,No,Yes,No,No,No,Very difficult,Yes,Yes,No,No,Yes,No,No,Yes,No comments
715,2014-08-28 10:07:00,<NA>,Male,United Kingdom,Not Available,No,No,Yes,Sometimes,26-100,No,No,No,No,No,No,Don't know,Somewhat difficult,Yes,No,No,No,No,Maybe,Don't know,No,No comments
734,2014-08-28 10:35:00,<NA>,Male,United States,OH,No,No,No,NaN,100-500,No,Yes,Don't know,Not sure,No,No,Don't know,Somewhat easy,No,No,Yes,Yes,No,No,Yes,No,We had a developer suffer from depression and pret
989,2014-08-29 09:10:00,<NA>,Others,Bahamas,IL,Yes,Yes,Yes,Often,1-5,Yes,Yes,Yes,Yes,Yes,Yes,Yes,Very easy,Yes,Yes,Yes,Yes,Yes,Yes,Yes,Yes,No comments
1090,2014-08-29 17:26:00,<NA>,Male,United States,OH,Yes,No,No,Never,1-5,Yes,Yes,No,Yes,No,No,Yes,Very easy,No,No,Some of them,Some of them,No,Maybe,Yes,No,No comments
1127,2014-08-30 20:55:00,<NA>,Others,United States,AL,Yes,Yes,Yes,Often,1-5,Yes,Yes,Yes,Yes,Yes,Yes,Yes,Very easy,Yes,Yes,Yes,Yes,Yes,Yes,Yes,Yes,password: testered


In [27]:
# Extract date and time from the original timestamp
df_survey["date"] = df_survey["timestamp"].dt.normalize()
df_survey["time"] = df_survey["timestamp"].dt.time

# Reorder columns: timestamp, date, time
cols = df_survey.columns.tolist()
cols.remove("date")
cols.remove("time")

timestamp_index = cols.index("timestamp")
cols[timestamp_index + 1:timestamp_index + 1] = ["date", "time"]

df_survey = df_survey[cols]

# Verify the result
display(df_survey[["timestamp", "date", "time"]].head())

,timestamp,date,time
0,2014-08-27 11:29:00,2014-08-27,11:29:00
1,2014-08-27 11:29:00,2014-08-27,11:29:00
2,2014-08-27 11:29:00,2014-08-27,11:29:00
3,2014-08-27 11:29:00,2014-08-27,11:29:00
4,2014-08-27 11:30:00,2014-08-27,11:30:00


In [28]:
df_survey.head()

,timestamp,date,time,age,gender,country,state,self_employed,family_history,treatment,work_interfere,no_employees,remote_work,tech_company,benefits,care_options,wellness_program,seek_help,anonymity,leave,mental_health_consequence,phys_health_consequence,coworkers,supervisor,mental_health_interview,phys_health_interview,mental_vs_physical,obs_consequence,comments
0,2014-08-27 11:29:00,2014-08-27,11:29:00,37,Female,United States,IL,No,No,Yes,Often,6-25,No,Yes,Yes,Not sure,No,Yes,Yes,Somewhat easy,No,No,Some of them,Yes,No,Maybe,Yes,No,No comments
1,2014-08-27 11:29:00,2014-08-27,11:29:00,44,Male,United States,IN,No,No,No,Rarely,More than 1000,No,No,Don't know,No,Don't know,Don't know,Don't know,Don't know,Maybe,No,No,No,No,No,Don't know,No,No comments
2,2014-08-27 11:29:00,2014-08-27,11:29:00,32,Male,Canada,Not Available,No,No,No,Rarely,6-25,No,Yes,No,No,No,No,Don't know,Somewhat difficult,No,No,Yes,Yes,Yes,Yes,No,No,No comments
3,2014-08-27 11:29:00,2014-08-27,11:29:00,31,Male,United Kingdom,Not Available,No,Yes,Yes,Often,26-100,No,Yes,No,Yes,No,No,No,Somewhat difficult,Yes,Yes,Some of them,No,Maybe,Maybe,No,Yes,No comments
4,2014-08-27 11:30:00,2014-08-27,11:30:00,31,Male,United States,TX,No,No,No,Never,100-500,Yes,Yes,Yes,No,Don't know,Don't know,Don't know,Don't know,No,No,Some of them,Yes,Yes,Yes,Don't know,No,No comments


In [29]:
# Identify exact duplicate records
duplicate_records = df_survey[df_survey.duplicated(keep=False)]

print("Number of duplicate occurrences:", df_survey.duplicated().sum())
print("Total rows involved in duplicates:", len(duplicate_records))

# Display all duplicate records
if not duplicate_records.empty:
    display(duplicate_records)
else:
    print("No duplicate records found.")

Number of duplicate occurrences: 1
Total rows involved in duplicates: 2


,timestamp,date,time,age,gender,country,state,self_employed,family_history,treatment,work_interfere,no_employees,remote_work,tech_company,benefits,care_options,wellness_program,seek_help,anonymity,leave,mental_health_consequence,phys_health_consequence,coworkers,supervisor,mental_health_interview,phys_health_interview,mental_vs_physical,obs_consequence,comments
859,2014-08-28 17:02:00,2014-08-28,17:02:00,32,Male,United Kingdom,Not Available,No,Yes,Yes,Rarely,6-25,No,No,No,No,No,No,No,Don't know,Yes,Maybe,Some of them,No,No,No,No,No,No comments
860,2014-08-28 17:02:00,2014-08-28,17:02:00,32,Male,United Kingdom,Not Available,No,Yes,Yes,Rarely,6-25,No,No,No,No,No,No,No,Don't know,Yes,Maybe,Some of them,No,No,No,No,No,No comments


In [30]:
# Remove exact duplicate records, keeping the first occurrence
rows_before = len(df_survey)

df = df_survey.drop_duplicates(keep="first").reset_index(drop=True)

rows_after = len(df)

print("Duplicate removal completed!")
print("Rows before:", rows_before)
print("Rows after:", rows_after)
print("Rows removed:", rows_before - rows_after)
print("Remaining duplicates:", df.duplicated().sum())

Duplicate removal completed!
Rows before: 1259
Rows after: 1258
Rows removed: 1
Remaining duplicates: 0


In [31]:
from pathlib import Path

# Navigate one level up from notebooks to the project root
output_dir = Path.cwd().parent / "data" / "processed"

# Ensure the directory exists
output_dir.mkdir(parents=True, exist_ok=True)

# Export duplicate-free modelling dataset
output_file = output_dir / "mental_health_survey_cleaned.csv"

df.to_csv(output_file, index=False, encoding="utf-8-sig")

# Confirm export
print("Dataset exported successfully!")
print(f"Location: {output_file.resolve()}")
print(f"Rows: {df.shape[0]}")
print(f"Columns: {df.shape[1]}")
print(f"File size: {output_file.stat().st_size / 1024:.2f} KB")
print(f"Remaining duplicates: {df.duplicated().sum()}")

Dataset exported successfully!
Location: <PROJECT_ROOT>\Documents\3rd Sem_porject Management and Data analytics lab\data\processed\mental_health_survey_cleaned.csv
Rows: 1258
Columns: 29
File size: 276.44 KB
Remaining duplicates: 0


In [32]:
df["work_interfere"].isna().sum()

np.int64(264)